# Assignment 2
2025 | IN6227 Data Mining | Roshani Ayu Pranasti | G2504973A

## Install Orange Association Library

- Documentation: https://orange3-associate.readthedocs.io/en/latest/
- GitHub: https://github.com/biolab/orange3-associate/tree/master

In [1]:
!pip3 install orange3 orange3-associate

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Import Libraries

In [ ]:
import pandas as pd
import time
from orangecontrib.associate.fpgrowth import *  # Association rule mining in Orange 3

## Dataset Definition
I have prepared 5 datasets of different sizes. All retrieved from Kaggle. More about the datasets can be read on `README.md`.
1. **Dataset 1**: `Datasets/Dataset1.csv`
2. **Dataset 2**: `Datasets/Dataset2.csv`
3. **Dataset 3**: `Datasets/Dataset3.csv`
4. **Dataset 4**: `Datasets/Dataset4.csv`
5. **Dataset 5**: `Datasets/Dataset5.csv`

## Data Preprocessing

### Dataset 1

In [3]:
# Load dataset
dataset1 = pd.read_csv("Datasets/Dataset1.csv")
display(dataset1)

,Transaction,Item,date_time,period_day,weekday_weekend
0,1,Bread,10/30/2016 9:58,morning,weekend
1,2,Scandinavian,10/30/2016 10:05,morning,weekend
2,2,Scandinavian,10/30/2016 10:05,morning,weekend
3,3,Hot chocolate,10/30/2016 10:07,morning,weekend
4,3,Jam,10/30/2016 10:07,morning,weekend
...,...,...,...,...,...
20502,9682,Coffee,4/9/2017 14:32,afternoon,weekend
20503,9682,Tea,4/9/2017 14:32,afternoon,weekend
20504,9683,Coffee,4/9/2017 14:57,afternoon,weekend
20505,9683,Pastry,4/9/2017 14:57,afternoon,weekend


In [4]:
print("Number of attributes in dataset 1:", dataset1.shape[1])
print("Number of data in dataset 1:", dataset1.shape[0], "\n")
dataset1.info()

Number of attributes in dataset 1: 5
Number of data in dataset 1: 20507 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Transaction      20507 non-null  int64 
 1   Item             20507 non-null  object
 2   date_time        20507 non-null  object
 3   period_day       20507 non-null  object
 4   weekday_weekend  20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [5]:
print("Number of unknown or missing values in dataset 1:")
dataset1.isnull().sum()

Number of unknown or missing values in dataset 1:


Transaction        0
Item               0
date_time          0
period_day         0
weekday_weekend    0
dtype: int64

In [6]:
# Check unique values for each column
for col in dataset1.columns:
    print(col, dataset1[col].unique())

Transaction [   1    2    3 ... 9682 9683 9684]
Item ['Bread' 'Scandinavian' 'Hot chocolate' 'Jam' 'Cookies' 'Muffin' 'Coffee'
 'Pastry' 'Medialuna' 'Tea' 'Tartine' 'Basket' 'Mineral water'
 'Farm House' 'Fudge' 'Juice' "Ella's Kitchen Pouches" 'Victorian Sponge'
 'Frittata' 'Hearty & Seasonal' 'Soup' 'Pick and Mix Bowls' 'Smoothies'
 'Cake' 'Mighty Protein' 'Chicken sand' 'Coke' 'My-5 Fruit Shoot'
 'Focaccia' 'Sandwich' 'Alfajores' 'Eggs' 'Brownie' 'Dulce de Leche'
 'Honey' 'The BART' 'Granola' 'Fairy Doors' 'Empanadas' 'Keeping It Local'
 'Art Tray' 'Bowl Nic Pitt' 'Bread Pudding' 'Adjustment' 'Truffles'
 'Chimichurri Oil' 'Bacon' 'Spread' 'Kids biscuit' 'Siblings'
 'Caramel bites' 'Jammie Dodgers' 'Tiffin' 'Olum & polenta' 'Polenta'
 'The Nomad' 'Hack the stack' 'Bakewell' 'Lemon and coconut' 'Toast'
 'Scone' 'Crepes' 'Vegan mincepie' 'Bare Popcorn' 'Muesli' 'Crisps'
 'Pintxos' 'Gingerbread syrup' 'Panatone' 'Brioche and salami'
 'Afternoon with the baker' 'Salad' 'Chicken Stew' 'Sp

#### Transform the Dataset to Encoded Transactions using One-Hot Encoding
For a simple association rule mining task, the goal is to find relationships between items within a transaction, regardless of when it happened. In this case, the time of day (`period_day`) or day of the week (`weekday_weekend`) is considered metadata about the transaction. This is why these columns were ignored when I converted the data to the required one-hot encoded format.

In [7]:
# Use get_dummies to one-hot encode the 'Item' column
one_hot_dataset1 = pd.get_dummies(dataset1["Item"])

# Combine it with the 'Transaction' column
one_hot_dataset1 = pd.concat([dataset1["Transaction"], one_hot_dataset1], axis=1)

# Group by transaction and sum the one-hot encoded columns
# This counts the occurrences of each item in each transaction
encoded_dataset1 = one_hot_dataset1.groupby('Transaction').sum()

# Convert counts to binary (0 or 1)
encoded_dataset1 = encoded_dataset1.map(lambda x: 1 if x > 0 else 0)

display(encoded_dataset1)
print("Encoded transactions dataset 1:\n", encoded_dataset1.values)

,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9681,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Encoded transactions dataset 1:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


### Dataset 2

In [8]:
# Load dataset
dataset2 = pd.read_csv("Datasets/Dataset2.csv")
display(dataset2)

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk
...,...,...,...
38760,4471,08-10-2014,sliced cheese
38761,2022,23-02-2014,candy
38762,1097,16-04-2014,cake bar
38763,1510,03-12-2014,fruit/vegetable juice


In [9]:
print("Number of attributes in dataset 2:", dataset2.shape[1])
print("Number of data in dataset 2:", dataset2.shape[0], "\n")
dataset2.info()

Number of attributes in dataset 2: 3
Number of data in dataset 2: 38765 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38765 entries, 0 to 38764
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Member_number    38765 non-null  int64 
 1   Date             38765 non-null  object
 2   itemDescription  38765 non-null  object
dtypes: int64(1), object(2)
memory usage: 908.7+ KB


In [10]:
print("Number of unknown or missing values in dataset 2:")
dataset2.isnull().sum()

Number of unknown or missing values in dataset 2:


Member_number      0
Date               0
itemDescription    0
dtype: int64

### Dataset 3

In [11]:
# Load dataset
dataset3 = pd.read_csv("Datasets/Dataset3.csv", sep=";")
display(dataset3)

/var/folders/rg/w9yw91kx569bmdd18wzk5wrr0000gn/T/ipykernel_86082/1320969518.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset3 = pd.read_csv("Datasets/Dataset3.csv", sep=";")


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
...,...,...,...,...,...,...,...
522059,581587,PACK OF 20 SPACEBOY NAPKINS,12,09.12.2011 12:50,"0,85",12680.0,France
522060,581587,CHILDREN'S APRON DOLLY GIRL,6,09.12.2011 12:50,"2,1",12680.0,France
522061,581587,CHILDRENS CUTLERY DOLLY GIRL,4,09.12.2011 12:50,"4,15",12680.0,France
522062,581587,CHILDRENS CUTLERY CIRCUS PARADE,4,09.12.2011 12:50,"4,15",12680.0,France


In [12]:
print("Number of attributes in dataset 3:", dataset3.shape[1])
print("Number of data in dataset 3:", dataset3.shape[0], "\n")
dataset3.info()

Number of attributes in dataset 3: 7
Number of data in dataset 3: 522064 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522064 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   BillNo      522064 non-null  object 
 1   Itemname    520609 non-null  object 
 2   Quantity    522064 non-null  int64  
 3   Date        522064 non-null  object 
 4   Price       522064 non-null  object 
 5   CustomerID  388023 non-null  float64
 6   Country     522064 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 27.9+ MB


In [13]:
print("Number of unknown or missing values in dataset 3:")
dataset3.isnull().sum()

Number of unknown or missing values in dataset 3:


BillNo             0
Itemname        1455
Quantity           0
Date               0
Price              0
CustomerID    134041
Country            0
dtype: int64

### Dataset 4

In [14]:
# Define column names
column_names = ["TV Show 1", "TV Show 2", "TV Show 3", "TV Show 4", "TV Show 5", "TV Show 6", "TV Show 7", "TV Show 8", "TV Show 9", "TV Show 10", "TV Show 11", "TV Show 12", "TV Show 13", "TV Show 14", "TV Show 15", "TV Show 16", "TV Show 17", "TV Show 18", "TV Show 19", "TV Show 20", "TV Show 21", "TV Show 22", "TV Show 23", "TV Show 24", "TV Show 25", "TV Show 26", "TV Show 27", "TV Show 28", "TV Show 29", "TV Show 30", "TV Show 31", "TV Show 32", ]

# Load dataset
dataset4 = pd.read_csv("Datasets/Dataset4.csv", header=None, names=column_names)
display(dataset4)

,TV Show 1,TV Show 2,TV Show 3,TV Show 4,TV Show 5,TV Show 6,TV Show 7,TV Show 8,TV Show 9,TV Show 10,...,TV Show 23,TV Show 24,TV Show 25,TV Show 26,TV Show 27,TV Show 28,TV Show 29,TV Show 30,TV Show 31,TV Show 32
0,Cobra Kai,Lupin,12 Monkeys,Sherlock,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Lost,Jack Ryan,The Flash,Game of thrones,House of Cards,12 Monkeys,Vikings,Fringe,The Mentalist,The Alienist,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sex Education,Dr. House,Kingdom,The Walking Dead,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ozark,Sex Education,Constantine,Preacher,Vikings,The Tick,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Naruto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9685,One Piece,The Blacklist,Two and a half men,Lupin,Dark,How I met your mother,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9686,One Piece,Mr. Robot,Succession,Ozark,12 Monkeys,Vikings,The Vampire Diaries,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9687,Daredevil,Atypical,Heros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9688,Absentia,The Newsroom,The Alienist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
print("Number of columns in dataset 4:", dataset4.shape[1])
print("Number of data in dataset 4:", dataset4.shape[0], "\n")

Number of columns in dataset 4: 32
Number of data in dataset 4: 9690 



In [16]:
print("Number of unknown or missing values in dataset 4:")
dataset4.isnull().sum()

Number of unknown or missing values in dataset 4:


TV Show 1        0
TV Show 2     2133
TV Show 3     3743
TV Show 4     5023
TV Show 5     6012
TV Show 6     6853
TV Show 7     7493
TV Show 8     8026
TV Show 9     8459
TV Show 10    8803
TV Show 11    9046
TV Show 12    9226
TV Show 13    9342
TV Show 14    9418
TV Show 15    9494
TV Show 16    9549
TV Show 17    9595
TV Show 18    9624
TV Show 19    9638
TV Show 20    9652
TV Show 21    9661
TV Show 22    9672
TV Show 23    9676
TV Show 24    9682
TV Show 25    9683
TV Show 26    9683
TV Show 27    9684
TV Show 28    9685
TV Show 29    9686
TV Show 30    9689
TV Show 31    9689
TV Show 32    9689
dtype: int64

### Dataset 5

In [17]:
# Load dataset
dataset5 = pd.read_csv("Datasets/Dataset5.csv")
display(dataset5)

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom
...,...,...,...,...,...,...,...,...
536345,C536548,12/1/2018,22168,Organiser Wood Antique White,18.96,-2,12472.0,Germany
536346,C536548,12/1/2018,21218,Red Spotty Biscuit Tin,14.09,-3,12472.0,Germany
536347,C536548,12/1/2018,20957,Porcelain Hanging Bell Small,11.74,-1,12472.0,Germany
536348,C536548,12/1/2018,22580,Advent Calendar Gingham Sack,16.35,-4,12472.0,Germany


In [18]:
print("Number of attributes in dataset 5:", dataset5.shape[1])
print("Number of data in dataset 5:", dataset5.shape[0], "\n")
dataset5.info()

Number of attributes in dataset 5: 8
Number of data in dataset 5: 536350 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  object 
 1   Date           536350 non-null  object 
 2   ProductNo      536350 non-null  object 
 3   ProductName    536350 non-null  object 
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  float64
 7   Country        536350 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.7+ MB


In [19]:
print("Number of unknown or missing values in dataset 5:")
dataset5.isnull().sum()

Number of unknown or missing values in dataset 5:


TransactionNo     0
Date              0
ProductNo         0
ProductName       0
Price             0
Quantity          0
CustomerNo       55
Country           0
dtype: int64

## Orange 3 Association Rule Mining

Based on its GitHub codebase, Orange 3 uses the FP-Growth (Frequent Pattern Growth) algorithm for its association rule mining.
- Orange 3 associate FP-growth: https://github.com/biolab/orange3-associate/blob/master/orangecontrib/associate/fpgrowth.py

### Dataset 1

In [25]:
min_support = 0.01
min_confidence = 0.4

# Create a mapping from column index to item name
mapping = {i: item_name for i, item_name in enumerate(encoded_dataset1.columns)}

# Find frequent itemsets
start_orange_association_rule = time.time()
itemsets = dict(frequent_itemsets(encoded_dataset1.values, min_support=min_support))

# Generate association rules from the frequent itemsets
rules = list(association_rules(itemsets, min_confidence=min_confidence))
end_orange_association_rule = time.time()

# Present processed rules as results
results = []
for antecedent, consequent, support, confidence in rules:
    antecedent_list = [mapping[item] for item in antecedent]
    consequent_list = [mapping[item] for item in consequent]

    results.append({
        "Antecedent": ", ".join(antecedent_list),
        "Consequent": ", ".join(consequent_list),
        "Support": support,
        "Confidence": confidence
    })

# Display results
orange_association_rule_time = end_orange_association_rule - start_orange_association_rule
print("Dataset 1 association rule mining time:", orange_association_rule_time)
results_df = pd.DataFrame(results)
print("Results:")
display(results_df)

# Sort the DataFrame by "Confidence" to see the most interesting rules first
results_df = results_df.sort_values(by="Confidence", ascending=False)
print("Top 5 with highest confidence:")
display(results_df.head(5))

Dataset 1 association rule mining time: 0.02729487419128418
Results:


,Antecedent,Consequent,Support,Confidence
0,"Bread, Cake",Coffee,95,0.429864
1,"Tea, Cake",Coffee,95,0.422222
2,Cake,Coffee,518,0.526958
3,Pastry,Coffee,450,0.552147
4,Sandwich,Coffee,362,0.532353
5,Medialuna,Coffee,333,0.569231
6,Hot chocolate,Coffee,280,0.507246
7,Cookies,Coffee,267,0.518447
8,Brownie,Coffee,186,0.490765
9,Juice,Coffee,195,0.534247


Top 5 with highest confidence:


,Antecedent,Consequent,Support,Confidence
14,Toast,Coffee,224,0.704403
15,Spanish Brunch,Coffee,103,0.598837
5,Medialuna,Coffee,333,0.569231
3,Pastry,Coffee,450,0.552147
11,Alfajores,Coffee,186,0.540698


## Brute-Force Approach